# ADDING FLAGS

In [16]:
import pandas as pd

# Load the dataset
file_path = "/content/ipl_bat_features.csv"
df = pd.read_csv(file_path)

# Ensure the 'date' column is in datetime format
df["date"] = pd.to_datetime(df["date"], errors='coerce')

# Columns that may have null values
stats_columns = [
    "career_avg", "career_rpm", "career_bpm",
    "inns_avg", "inns_rpm", "inns_bpm",
    "ven_avg", "ven_rpm", "ven_bpm",
    "ssn_avg", "ssn_rpm", "ssn_bpm", "career_sr", "career_bfpm",
    "inns_sr", "inns_bfpm",
    "ven_sr", "ven_bfpm",
    "ssn_sr", "ssn_bfpm"
]
last_5_matches_columns = [
    "last5_avg", "last5_rpm", "last5_bpm", "last5_sr", "last5_bfpm"
]
# Identifying debuts
# Career debut: First match of the player based on earliest date
debut_dates = df.groupby("batter_id")["date"].transform("min")
df["career_debut"] = (df["date"] == debut_dates).astype(int)

# Season debut: First match of the player in a season
df["season_debut"] = (df.groupby(["batter_id", df["date"].dt.year])["date"].transform("min") == df["date"]).astype(int)

# Venue debut: First match of the player at a venue
df["venue_debut"] = (df.groupby(["batter_id", "venue"])["date"].transform("min") == df["date"]).astype(int)

# Innings debut: First time the player is batting in a specific innings (1 or 2) across all matches
df["innings_debut"] = (df.groupby(["batter_id", "inns"])["date"].transform("min") == df["date"]).astype(int)



# ADDING 0 TO NULL VALUES


In [17]:
# Fill null values with 0 if it's any kind of debut
df.loc[df[["career_debut", "season_debut", "venue_debut", "innings_debut"]].any(axis=1), stats_columns] = df[stats_columns].fillna(0)


# Fill null values in last 5 matches columns with 0 only if it's a career debut
df.loc[df["career_debut"] == 1, last_5_matches_columns] = df[last_5_matches_columns].fillna(0)

# Save the cleaned data
output_path = "null_with_0_cleaned_ipl_bat_features.csv"
df.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")


Cleaned data saved to: null_with_0_cleaned_ipl_bat_features.csv
